# Volatility Surface Modelling for Crypto Options
### A functional library for fitting, comparing and serving implied-volatility surfaces

**Natixis CIB — Quant Analyst Internship**

---

This notebook is a start-to-finish walkthrough of the `volatility_surface` library.
It doubles as the narrative of the project: each section introduces a piece of the
problem, shows the tool we built for it, and — where relevant — the **trials and
errors** that shaped the final design.

By the end you will be able to:

1. Load and clean a Deribit option snapshot into a calibration-ready frame.
2. Instantiate and inspect the four smile models (**Raw SVI, SSVI, eSSVI, SABR**).
3. Calibrate a single slice and a full surface, choosing a fitting **objective**.
4. Compare models on held-out metrics and **check for static arbitrage**.
5. Reproduce the FX-style **Reiswich–Wystup** parabolic smile and see why it
   under-fits crypto wings.
6. Price options under **Black-76 / Black-Scholes / Garman-Kohlhagen** and recover
   the implied funding rate of the inverse contracts.
7. Run the **fast recalibration** path that powers the live trading screen.

> **Runtime note.** Most cells run in well under a second. The three *global*
> calibrations (eSSVI / SSVI / SABR) take **~1–2 minutes each** on a single
> snapshot — they are flagged inline. Run the notebook top-to-bottom once.

## Table of contents

0. [Environment setup](#sec0)
1. [The data pipeline: Deribit → parquet](#sec1)
2. [The smile models](#sec2)
3. [Single-slice calibration](#sec3)
4. [Objective functions & the vega-weighting story](#sec4)
5. [Global calibration (eSSVI / SSVI / SABR)](#sec5)
6. [Model comparison](#sec6)
7. [Static-arbitrage checking](#sec7)
8. [The Reiswich–Wystup parabolic detour](#sec8)
9. [Pricing models & inverse options](#sec9)
10. [Fast recalibration (production speed)](#sec10)
11. [The live system](#sec11)
12. [Trials & errors](#sec12)
13. [Conclusion & next steps](#sec13)

<a id="sec0"></a>
## 0. Environment setup

We locate the project root (the folder containing `2 - Data`), put it on the
import path, and pull in every public entry point we will use.

In [ ]:
import os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 110})

# Walk up until we find the project root (contains "2 - Data")
here = Path.cwd()
PROJECT_ROOT = next((p for p in [here, *here.parents] if (p / "2 - Data").exists()), here)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

PARQUET_ETH = PROJECT_ROOT / "2 - Data" / "parquets" / "eth_options_data_cleaned.parquet"
PARQUET_BTC = PROJECT_ROOT / "2 - Data" / "parquets" / "btc_options_data_cleaned.parquet"
print("ETH archive exists:", PARQUET_ETH.exists())

In [ ]:
# Library entry points
from volatility_surface.models.vol_models import get_model, MODELS, RawSVI, SSVI, SABR, eSSVI
from volatility_surface.core.calibration.calibrator import (
    load_snapshot, calibrate_snapshot,
    calibrate_global_essvi, calibrate_global_ssvi, calibrate_global_sabr,
    calibrate_global_essvi_update, calibrate_snapshot_sabr_update,
    summary, compare, crossedness,
)
from volatility_surface.core.calibration.objectives import (
    get_objective, OBJECTIVES, evaluate_all, DEFAULT_METRICS, optimize_beta,
)
from volatility_surface.plots.vol_plots import (
    plot_all, plot_slices, plot_surface, plot_total_variance,
    plot_metrics, plot_compare,
)
from volatility_surface.utils.arbitrage_checker import check_arbitrage, plot_arbitrage_map
from volatility_surface.models.rw_parabolic import (
    fit_paper_style, fit_least_squares, sigma_of_K, RWParams,
)
from volatility_surface.core.pricing.pricing_models import bs_price, gk_price

print("Models available :", list(MODELS.keys()))
print("Objectives       :", list(OBJECTIVES.keys()))
print("Default metrics  :", DEFAULT_METRICS)

<a id="sec1"></a>
## 1. The data pipeline: Deribit → parquet

Crypto option quotes come from **Deribit**, the dominant venue for BTC/ETH options.
The raw feed is a per-instrument book summary (one row per listed option). Our
offline pipeline (`utils/data_handling.clean_df`) and the live gatherer
(`live_gather.clean_live`) both turn that into a **calibration-ready frame** with:

| column | meaning |
|---|---|
| `k` | log-moneyness $\ln(K/F)$ |
| `t` | time to expiry in years |
| `w` | total variance $\sigma_{\text{mark}}^2\,t$ |
| `mark_iv` | Deribit mark implied vol |
| `bid_iv` / `ask_iv` | implied vols inverted from bid/ask **prices** (via `vollib`) |
| `vega` | analytic Black vega, used as the default fitting weight |
| `underlying_price` | per-expiry **forward** $F$ |
| `estimated_delivery_price` | **spot** index $S$ |

The cleaning enforces an **OTM-only** filter for calibration (calls with $k\ge 0$,
puts with $k\le 0$) so that put–call parity duplicates don't double-weight strikes.

In [ ]:
raw = pd.read_parquet(PARQUET_ETH)
print(f"ETH archive: {len(raw):,} rows across {raw['file_timestamp'].nunique()} snapshots")
print("Columns:", list(raw.columns))
raw[["file_timestamp", "strike", "option_type", "t", "k",
     "mark_iv", "bid_iv", "ask_iv", "vega", "underlying_price"]].head()

In [ ]:
# load_snapshot() pulls a single timestamp and re-derives k, w, t cleanly.
snapshots = sorted(raw["file_timestamp"].unique())
SNAP_TS = snapshots[0]
df = load_snapshot(str(PARQUET_ETH), timestamp=SNAP_TS)

expiries = sorted(df["t"].unique())
print(f"Snapshot {SNAP_TS}: {len(df)} quotes, {len(expiries)} expiries")
print("Expiries (days):", [round(t*365, 1) for t in expiries])

In [ ]:
# OTM-only view used for all calibration below
otm = ((df["option_type"] == "C") & (df["k"] >= 0)) | \
      ((df["option_type"] == "P") & (df["k"] <= 0))
df_otm = df[otm].copy()
print(f"Full snapshot: {len(df)} quotes  ->  OTM-only: {len(df_otm)} quotes")

# Quick look at one expiry's smile
T0 = expiries[len(expiries)//2]
sl = df_otm[np.isclose(df_otm["t"], T0)].sort_values("k")
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(sl["k"], sl["mark_iv"], s=18, c="#6A0DAD", label="mark IV")
ax.fill_between(sl["k"], sl["bid_iv"], sl["ask_iv"], alpha=0.2, color="#B284E0", label="bid-ask band")
ax.set(xlabel="log-moneyness k", ylabel="implied vol",
       title=f"ETH market smile  T = {T0*365:.0f}d")
ax.legend(); plt.show()

<a id="sec2"></a>
## 2. The smile models

Every model subclasses `VolModel` and implements `w(k, params)` — the total
variance. Implied vol is `iv = sqrt(w / t)`, and the risk-neutral density comes
from `butterfly_density(k, params)` (Breeden–Litzenberger).

| model | per-slice params | global params | $w(k)$ form |
|---|---|---|---|
| **Raw SVI** | $a, b, \rho, m, \sigma$ | – | $a + b[\rho(k-m)+\sqrt{(k-m)^2+\sigma^2}]$ |
| **SSVI** | $\theta$ | $\eta,\gamma,\rho$ | Gatheral–Jacquier surface |
| **eSSVI** | $\theta,\eta,\gamma$ | $\rho_0,\rho_\infty,\lambda$ | $\rho(\theta)=\rho_\infty+(\rho_0-\rho_\infty)e^{-\lambda\theta}$ |
| **SABR** | $\alpha,\rho,\nu$ ($\beta$ fixed) | – | Hagan (2002) lognormal expansion |

The registry `get_model("svi" | "ssvi" | "essvi" | "sabr", **kwargs)` returns an
instance.

In [ ]:
for name in ["svi", "ssvi", "essvi", "sabr"]:
    m = get_model(name)
    k = sl["k"].values
    w_obs = sl["w"].values
    guess = m.initial_guess(k, w_obs, T0)
    print(f"{m.name:>10s}  initial guess: { {kk: round(v,3) for kk,v in guess.items()} }")

### Drawing a smile and its risk-neutral density from hand-set parameters

A `VolModel` is a pure function of its parameter dict — handy for intuition. Here is
a Raw SVI slice and the implied density it generates (negative density = butterfly
arbitrage).

In [ ]:
svi = RawSVI()
params = {"a": 0.04, "b": 0.40, "rho": -0.30, "m": 0.05, "sig": 0.20}
kk = np.linspace(-1.0, 1.0, 401)
iv = svi.iv(kk, params, t=0.25)
dens = svi.butterfly_density(kk, params)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(kk, iv, c="#6A0DAD", lw=2.2); a1.set(xlabel="k", ylabel="IV", title="Raw SVI smile")
a2.fill_between(kk, np.maximum(dens, 0), color="#C9A0DC", alpha=0.7)
a2.plot(kk, dens, c="#6A0DAD", lw=1.5); a2.axhline(0, c="#999", lw=0.6)
a2.set(xlabel="k", ylabel="q(k)", title="Risk-neutral density")
print("Butterfly-arbitrage free:", not svi.has_butterfly_arb(params),
      "(min g =", round(svi.min_g(params), 4), ")")
plt.show()

<a id="sec3"></a>
## 3. Single-slice calibration

`calibrate_snapshot(df, model, objective, ...)` fits every expiry independently
(for per-slice models like Raw SVI and SABR). It runs a global differential-evolution
search followed by a Nelder–Mead polish, with **soft penalties** for negative
variance, butterfly arbitrage, and calendar crossing with neighbouring slices.

We pass the bid/ask IV columns so liquidity-aware metrics like `spread_hit_rate`
(fraction of fitted IVs landing inside the quoted band) can be evaluated.

In [ ]:
# Raw SVI, per-slice. (~30-60 s: DE + NM on each expiry.)
t0 = time.perf_counter()
res_svi = calibrate_snapshot(
    df_otm, model=RawSVI(), objective="vega_wmse",
    vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
)
print(f"Calibrated {len(res_svi['expiries'])} slices in {time.perf_counter()-t0:.1f}s")
summary(res_svi)

In [ ]:
# Per-slice fitted smiles with the market scatter + the implied RND on each.
fig = plot_slices(res_svi, df=df_otm)
plt.show()

<a id="sec4"></a>
## 4. Objective functions & the vega-weighting story

The **objective** decides *which strikes the fit cares about*. All objectives share
the signature `f(w_fit, w_obs, iv_fit, iv_obs, k, t, spreads=...)` and are looked up
by name from `OBJECTIVES`.

Key members:

- `iv_mse` — unweighted (every quote equal; over-weights illiquid wings).
- `vega_wmse` — **vega-weighted** MSE in vol space ($\propto$ vega$^2/$spread$^2$): the
  FactSet-style default. ATM-dominated and liquidity-aware.
- `iv_zweighted(β)` — vega weighting with a standardised-moneyness boost
  $\exp(\beta z^2)$ that controls how much the wings count. $\beta=0$ is pure vega,
  $\beta=0.5$ ≈ uniform-in-$z$.
- `band` / `vega_wmse_band` — explicitly penalise leaving the bid-ask book.

> **Trial & error.** On a handful of ETH dates, `iv_zweighted(β=0.4)` looked best for
> `spread_hit_rate`. But across *more* snapshots the advantage evaporated and plain
> **vega-weighting (β=0)** was the robust choice — so the live system uses
> `vega_wmse`. The lesson: one snapshot is an anecdote, not a result.

In [ ]:
# Score a fitted result on every default metric (locate the slice by nearest T).
i_svi = int(np.argmin(np.abs(np.asarray(res_svi["expiries"]) - T0)))
p_svi = res_svi["params"][i_svi]
mdl   = res_svi["_model"]
evals = evaluate_all(
    w_fit=mdl.w(sl["k"].values, p_svi),
    w_obs=sl["w"].values,
    iv_fit=mdl.iv(sl["k"].values, p_svi, T0),
    iv_obs=sl["mark_iv"].values,
    k=sl["k"].values, t=T0,
    bid=sl["bid_iv"].values, ask=sl["ask_iv"].values,
)
pd.Series(evals).round(4).to_frame("value (mid expiry, Raw SVI)")

In [ ]:
# Overlay three objectives on a SINGLE expiry to see the weighting effect.
results_obj = []
for obj in ["iv_mse", "vega_wmse", "iv_zweighted"]:
    r = calibrate_snapshot(
        df_otm[np.isclose(df_otm["t"], T0)], model=RawSVI(), objective=obj,
        vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
    )
    r["objective"] = obj
    results_obj.append(r)

fig = plot_compare(results_obj, df=df_otm[np.isclose(df_otm["t"], T0)], t_exp=T0)
plt.show()

### The `optimize_beta` sweep (optional, slow)

`optimize_beta` grids the wing-emphasis parameter and reports the target metric per
β. It re-runs a full calibration for each β, so it is **slow** — the cell is left for
you to run. Our finding: the metric is nearly flat in β on the aggregate, with β=0
the safest pick.

In [ ]:
# Uncomment to run (~2-4 min: one full per-slice calibration per beta).
# beta_out = optimize_beta(
#     df_otm, calibrate_fn=calibrate_snapshot,
#     calibrate_kwargs=dict(model=RawSVI(), vega_col="vega",
#                           bid_col="bid_iv", ask_col="ask_iv", verbose=False),
#     beta_grid=[0.0, 0.2, 0.4], target_metric="spread_hit_rate", plot=True,
# )
print("optimize_beta is available — uncomment above to run the sweep.")

<a id="sec5"></a>
## 5. Global calibration (eSSVI / SSVI / SABR)

Per-slice fits are flexible but can **cross in maturity** (calendar arbitrage).
Global models share parameters across expiries to enforce a coherent surface.

- `calibrate_global_essvi` — 3 global $(\rho_0,\rho_\infty,\lambda)$ + 3 per slice.
- `calibrate_global_ssvi` — 3 global $(\eta,\gamma,\rho)$ + 1 per slice; provably
  arb-free under the GJ conditions.
- `calibrate_global_sabr` — per-slice SABR with calendar coupling.

> **Runtime: ~1–2 minutes each** on this snapshot.

In [ ]:
t0 = time.perf_counter()
res_essvi = calibrate_global_essvi(
    df_otm, model=eSSVI(), objective="vega_wmse",
    vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
)
print(f"eSSVI global fit: {time.perf_counter()-t0:.1f}s")
summary(res_essvi)

In [ ]:
t0 = time.perf_counter()
res_ssvi = calibrate_global_ssvi(
    df_otm, objective="vega_wmse",
    vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
)
print(f"SSVI global fit: {time.perf_counter()-t0:.1f}s")

t0 = time.perf_counter()
res_sabr = calibrate_global_sabr(
    df_otm, objective="vega_wmse",
    vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
)
print(f"SABR global fit: {time.perf_counter()-t0:.1f}s")

In [ ]:
# eSSVI fitted surface (left: smile heatmap; right: term structure of theta)
fig = plot_surface(res_essvi)
plt.show()

<a id="sec6"></a>
## 6. Model comparison

`compare(*results)` aligns the per-slice metrics into one table. More parameters →
better in-sample fit, but watch the calendar/butterfly columns: Raw SVI's flexibility
is exactly what makes it prone to arbitrage between expiries.

In [ ]:
cmp = compare(res_svi, res_essvi, res_ssvi, res_sabr)
cmp

In [ ]:
# Overlay all four fitted models on one expiry.
for r, nm in [(res_svi, "Raw SVI"), (res_essvi, "eSSVI"),
              (res_ssvi, "SSVI"), (res_sabr, "SABR")]:
    r["model_name"] = nm
fig = plot_compare([res_svi, res_essvi, res_ssvi, res_sabr],
                   df=df_otm, t_exp=T0)
plt.show()

### Aggregate benchmark across snapshots

The standalone script `benchmark_objectives.py` sweeps ~10 objectives on global
eSSVI across **all** archived snapshots. If you have run it, the pickle is loaded
below; otherwise this cell is a no-op.

In [ ]:
bench_pkl = PROJECT_ROOT / "benchmark_objectives_essvi.pkl"
if bench_pkl.exists():
    import pickle
    with open(bench_pkl, "rb") as f:
        b = pickle.load(f)
    display(b["summary"].round(4))
else:
    print("Run `python volatility_surface/benchmark_objectives.py` to generate this.")

<a id="sec7"></a>
## 7. Static-arbitrage checking

`check_arbitrage` evaluates the fitted surface on a dense $(k, t)$ grid and tests the
two Gatheral–Jacquier conditions:

- **Calendar:** $\partial_t w \ge 0$ (total variance non-decreasing in maturity).
- **Butterfly:** $g(k, t) \ge 0$ (non-negative risk-neutral density).

`plot_arbitrage_map` renders a 2×2 diagnostic with violations overlaid in red.

In [ ]:
arb = check_arbitrage(res_essvi)
print(f"Calendar  free: {arb['calendar_free']}   "
      f"worst dw/dt = {arb['calendar_worst']:+.3e}   "
      f"violations = {arb['calendar_violation_pct']:.2f}% of grid")
print(f"Butterfly free: {arb['butterfly_free']}   "
      f"worst g     = {arb['butterfly_worst']:+.3e}   "
      f"violations = {arb['butterfly_violation_pct']:.2f}% of grid")

In [ ]:
fig = plot_arbitrage_map(res_essvi)
plt.show()

<a id="sec8"></a>
## 8. The Reiswich–Wystup parabolic detour

The FX literature (Reiswich & Wystup, 2010) builds the smile as a **parabola in
delta space**, anchored on three quotes: ATM vol, the 25Δ risk reversal, and the
25Δ strangle. We implemented it in `models/rw_parabolic.py` with two fits:

- `fit_paper_style` — the faithful 3-point extraction.
- `fit_least_squares` — the same 3-parameter form, fit to the whole chain.

> **Finding.** It nails ATM precision by construction but **cannot bend hard enough
> for crypto wings** — on Deribit it is 3–10× worse than SABR (same parameter count).
> Right tool for a 3-quote FX market; wrong tool for a 30-strike crypto chain.

In [ ]:
F = float(sl["underlying_price"].iloc[0])
k_obs, iv_obs = sl["k"].values, sl["mark_iv"].values

p_paper = fit_paper_style(k_obs, iv_obs, T0, F)
p_ls    = fit_least_squares(k_obs, iv_obs, T0, F)
print("paper-style:", p_paper)
print("LS-refined :", p_ls)

K = F * np.exp(k_obs)
rmse = lambda p: np.sqrt(np.mean((sigma_of_K(K, F, T0, p.sigma_atm, p.sigma_rr, p.sigma_s) - iv_obs)**2))
print(f"\nRW paper-style IV RMSE: {rmse(p_paper)*100:.2f} vol-pts")
print(f"RW LS-refined  IV RMSE: {rmse(p_ls)*100:.2f} vol-pts")

In [ ]:
Kg = F * np.exp(np.linspace(k_obs.min(), k_obs.max(), 200))
kg = np.log(Kg / F)
i_e = int(np.argmin(np.abs(np.asarray(res_essvi["expiries"]) - T0)))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.scatter(k_obs, iv_obs, s=16, c="#444", label="market", zorder=4)
ax.plot(kg, res_essvi["_model"].iv(kg, res_essvi["params"][i_e], T0), c="#1F77B4", lw=2, label="eSSVI")
ax.plot(kg, sigma_of_K(Kg, F, T0, p_paper.sigma_atm, p_paper.sigma_rr, p_paper.sigma_s), "--", c="#9A6BB5", lw=2, label="RW paper")
ax.plot(kg, sigma_of_K(Kg, F, T0, p_ls.sigma_atm, p_ls.sigma_rr, p_ls.sigma_s), c="#6A0DAD", lw=2, label="RW LS")
ax.set(xlabel="k", ylabel="IV", title=f"RW parabolic vs eSSVI  T={T0*365:.0f}d")
ax.legend(); plt.show()

<a id="sec9"></a>
## 9. Pricing models & inverse options

Three engines share the same Black skeleton and differ only in the carry term:

- **Black-76** — forward measure, no rates.
- **Black-Scholes** — drift $r$.
- **Garman-Kohlhagen** — drift $r - q$ ($q$ = funding/staking/basis for crypto).

Deribit options are **inverse**: premiums are quoted in the coin, not USD. The venue
publishes the **spot** $S$ (`estimated_delivery_price`) and the per-expiry **forward**
$F$ (`underlying_price`); the implied interest rate is $r = \ln(F/S)/t$.

In [ ]:
# Implied rate term structure from the snapshot
rate_rows = []
for t in expiries:
    s = df[np.isclose(df["t"], t)]
    F_t = float(s["underlying_price"].iloc[0])
    S_t = float(s["estimated_delivery_price"].iloc[0]) if "estimated_delivery_price" in s else F_t
    rate_rows.append({"days": round(t*365, 1), "S": S_t, "F": F_t,
                      "basis_%": (F_t/S_t - 1)*100, "implied_r_%": np.log(F_t/S_t)/t*100})
rates = pd.DataFrame(rate_rows)
display(rates.round(3))

In [ ]:
# Same IV, three engines: B76 == BS(r=0) == GK(r=q=0); rates shift the price.
S, Kx, T, sigma = 2000.0, 2100.0, 0.25, 0.55
print("Black-76 (r=q=0):", round(float(bs_price(S, Kx, T, 0.0, sigma, "c")), 4))
print("BS (r=4%)       :", round(float(bs_price(S, Kx, T, 0.04, sigma, "c")), 4))
print("GK (r=4%, q=1%) :", round(float(gk_price(S, Kx, T, 0.04, 0.01, sigma, "c")), 4))

<a id="sec10"></a>
## 10. Fast recalibration (production speed)

The live screen cannot wait 50 s per tick. Two **warm-start** paths reuse the previous
fit:

- `calibrate_global_essvi_update(mode="theta_only")` — freezes every shape parameter
  and refits only the per-slice ATM levels $\theta_i$ via 1-D Brent searches.
  **~30 ms.**
- `calibrate_snapshot_sabr_update` — per-slice Nelder–Mead from the previous SABR
  params, no DE. **~5 s** (looser tolerances than the cold fit).

Below we time the eSSVI fast path against itself on a neighbouring snapshot.

In [ ]:
SNAP_TS2 = snapshots[1]
df2 = load_snapshot(str(PARQUET_ETH), timestamp=SNAP_TS2)
otm2 = ((df2["option_type"] == "C") & (df2["k"] >= 0)) | \
       ((df2["option_type"] == "P") & (df2["k"] <= 0))
df2_otm = df2[otm2].copy()

t0 = time.perf_counter()
res_fast = calibrate_global_essvi_update(
    df2_otm, prev_result=res_essvi, model=res_essvi["_model"],
    objective="vega_wmse", mode="theta_only",
    vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
)
dt = (time.perf_counter() - t0) * 1000
print(f"eSSVI theta-only update on the next snapshot: {dt:.1f} ms "
      f"({len(res_fast['expiries'])} slices)")

In [ ]:
# SABR warm-start update (~5 s) — needs a SABR seed.
t0 = time.perf_counter()
res_sabr_fast = calibrate_snapshot_sabr_update(
    df2_otm, prev_result=res_sabr, objective="vega_wmse",
    vega_col="vega", bid_col="bid_iv", ask_col="ask_iv", verbose=False,
)
print(f"SABR warm-start update: {time.perf_counter()-t0:.1f}s  "
      f"(vs ~50 s cold)")

<a id="sec11"></a>
## 11. The live system

Everything above is wired into a **per-second trading screen**. Architecture:

```
   Deribit REST  ──┐        Deribit WS (ticker.*)
   (book summary)  │              (top-of-book sizes)
                   ▼                     │
        live_gather.py  ◀───────────────┘
          clean_live() → OTM filter → IV inversion
          atomic append → rolling parquet (60-min window)
                   │
                   ▼
        streamlit_chain.py
          read latest snapshot
          first tick : full eSSVI / SABR  (cold, cached)
          each tick  : theta-only / warm-start update
          render option chain  +  maturity tabs  +  arb banner
```

Launch both gatherers (ETH + BTC) and the UI with one command:

```bash
./run_live.sh           # both currencies, 1 s polling
```

The screen mirrors a Deribit-style chain: invariant greeks on the left, the smile
vol, call/put markets either side of a black **STRIKE** column (ATM highlighted
yellow), bid/ask **sizes** from the websocket, and theo prices that turn **red** when
they fall outside the quoted spread. A top banner flags any **calendar-spread
arbitrage** detected across the fitted expiries.

*(These run as separate processes — not executed here.)*

In [ ]:
# Sanity: the calendar-arb detector the live banner uses.
exp_arr = np.asarray(res_essvi["expiries"])
order = np.argsort(exp_arr)
worst = 0.0
for i in range(len(order)-1):
    a, b = order[i], order[i+1]
    worst = max(worst, crossedness(res_essvi["_model"],
                                   res_essvi["params"][a], res_essvi["params"][b]))
print(f"Worst calendar crossing in the eSSVI surface: {worst:.3e}",
      "(>1e-6 would flag the red banner)")

<a id="sec12"></a>
## 12. Trials & errors

The honest version of the project — what broke and what we learned:

1. **β tuning over-fit on few dates.** `iv_zweighted(β=0.4)` won on ~10 ETH dates;
   the edge vanished on the full set. Settled on plain **vega-weighting**.
2. **SVI-JW degeneracy.** With $\rho\approx 0$ and $\psi\approx 0$, the JW→raw map is
   singular; a tiny skew change produced a huge $\sigma$ and a flat smile. Fixed with
   an explicit degeneracy guard + UI warning.
3. **Inverse-option units.** Black-76 returns USD; Deribit quotes in the coin. Every
   theo had to be divided by $F$ — otherwise the whole chain flagged false arbitrage.
4. **IV inversion returned `None`.** `implied_volatility_dataframe` returned a scalar
   `None` on invalid quotes, which collapsed the `df[["bid_iv","ask_iv"]]` assignment
   with *"Columns must be same length as key"* and froze the gatherer. Fixed by always
   returning a `(nan, nan)` pair.
5. **Streamlit `@st.cache_resource` evicts on file save.** Switching models forced a
   50 s recompute. Fixed by keeping the warm-start fit in `st.session_state` keyed by
   `(currency, model)`.
6. **Hard-coded currency.** The app always read `live_eth.parquet`; selecting BTC
   showed stale data. Fixed with a per-currency parquet resolver + auto-detect of the
   freshest feed.
7. **One snapshot is not a result.** The recurring theme — every promising single-snapshot
   finding had to survive a multi-snapshot benchmark before we trusted it.

<a id="sec13"></a>
## 13. Conclusion & next steps

We built an end-to-end, **arbitrage-aware** volatility-surface library for crypto
options: a cleaning pipeline, four calibratable smile models, a registry of
liquidity-aware objectives, static-arbitrage diagnostics, and a per-second live
trading screen powered by warm-start recalibration.

**What worked.** eSSVI with vega-weighting gives a coherent, near-arbitrage-free
surface; the theta-only fast path keeps it live at 30 ms/tick. SABR is the best
3-parameter per-slice baseline.

**Open questions / next steps.**
- Parallelise the SABR warm-start (per-slice, embarrassingly parallel) to reach 1 Hz.
- Regime-dependent objective or smooth $\beta(\text{spread})$ instead of a hard switch.
- Re-invert bid/ask IVs under each pricing engine for a fully consistent model bake-off.
- Extend the RW parabola (cubic/quartic in delta) to test whether the wing miss is one
  free parameter away from being fixed.

---
*End of walkthrough.*